# 03 – Tập vàng: lấy mẫu, gán nhãn tay, kiểm tra

**Phần 1** (chạy một lần): lấy 500 câu từ kho vàng, phân tầng theo sao, dư lớp 1–3 sao. Tạo file để **bạn tự gán nhãn**, đã ẩn số sao và tên sản phẩm.

**Phần 2** (chạy sau khi gán xong): kiểm tra file nhãn, bảng chéo nhãn tay × số sao, Cohen's kappa.

In [1]:
import os
import pandas as pd
from sklearn.metrics import cohen_kappa_score

LAB = "../labeling"
os.makedirs(LAB, exist_ok=True)
pool = pd.read_csv("../data/processed/shopee_gold_pool.csv")
print(len(pool)); print(pool.rating.value_counts().sort_index())

746
rating
1    100
2     60
3     50
4    166
5    370
Name: count, dtype: int64


## Phần 1. Lấy mẫu 500 câu

Lấy **toàn bộ** câu 1–3 sao (lớp neg/neu hiếm), phần còn lại lấy ngẫu nhiên trong 4–5 sao. Xáo trộn thứ tự để khi gán không đoán được sao. 200 câu đầu gán cả khía cạnh.

Cell này **không ghi đè** nếu file đã có (để khỏi mất nhãn bạn đã gán).

In [2]:
N = 500
TO_LABEL = f"{LAB}/gold_500_to_label.csv"
KEY = f"{LAB}/gold_500_key.csv"

if os.path.exists(TO_LABEL):
    print("Đã có", TO_LABEL, "-> bỏ qua, không ghi đè.")
else:
    low = pool[pool.rating <= 3]
    high = pool[pool.rating >= 4].sample(N - len(low), random_state=42)
    sample = pd.concat([low, high]).sample(frac=1, random_state=42).reset_index(drop=True)
    sample["stt"] = range(1, len(sample) + 1)
    sample["gan_khia_canh"] = (sample.stt <= 200).map({True: "có", False: ""})
    sheet = sample[["stt", "id", "gan_khia_canh", "text_clean"]].rename(columns={"text_clean": "text"})
    for col in ["sentiment", "giao_hang", "chat_luong", "gia_ca", "dich_vu", "ghi_chu"]:
        sheet[col] = ""
    sheet.to_csv(TO_LABEL, index=False, encoding="utf-8-sig")          # utf-8-sig để Excel đọc đúng tiếng Việt
    sample[["id", "product", "rating", "weak_label"]].to_csv(KEY, index=False, encoding="utf-8-sig")
    print("Đã tạo", TO_LABEL, "và", KEY)

key = pd.read_csv(KEY)
print(key.rating.value_counts().sort_index())

Đã tạo ../labeling/gold_500_to_label.csv và ../labeling/gold_500_key.csv
rating
1    100
2     60
3     50
4     95
5    195
Name: count, dtype: int64


In [3]:
# File gán lại 60 câu để tính kappa (intra-annotator). Gán file này SAU, không xem nhãn cũ.
RELABEL = f"{LAB}/gold_relabel_60.csv"
if not os.path.exists(RELABEL):
    first150 = pd.read_csv(TO_LABEL).head(150)
    first150.sample(60, random_state=7)[["id", "text"]].assign(sentiment="").to_csv(
        RELABEL, index=False, encoding="utf-8-sig")
    print("Đã tạo", RELABEL)

Đã tạo ../labeling/gold_relabel_60.csv


## Phần 2. Sau khi gán xong

Gán trong `gold_500_to_label.csv` rồi lưu thành **`labeling/gold_500.csv`** (Excel: *Save As → CSV UTF-8*). Giá trị hợp lệ: `sentiment` ∈ pos/neg/neu; 4 cột khía cạnh ∈ pos/neg/neu/none (chỉ với dòng `gan_khia_canh = có`).

In [4]:
GOLD = f"{LAB}/gold_500.csv"
ASPECTS = ["giao_hang", "chat_luong", "gia_ca", "dich_vu"]

if not os.path.exists(GOLD):
    print("Chưa có", GOLD, "— gán nhãn xong rồi chạy lại từ đây.")
else:
    gold = pd.read_csv(GOLD).merge(key, on="id")
    for c in ["sentiment"] + ASPECTS:
        gold[c] = gold[c].astype("string").str.strip().str.lower()
    done = gold.sentiment.notna()
    print(f"Đã gán cảm xúc: {done.sum()}/{len(gold)}")
    bad = gold[done & ~gold.sentiment.isin(["pos", "neg", "neu"])]
    print("Giá trị cảm xúc sai:", len(bad)); display(bad[["stt", "sentiment"]].head())
    asp_rows = gold[gold.gan_khia_canh == "có"]
    for a in ASPECTS:
        wrong = asp_rows[a].notna() & ~asp_rows[a].isin(["pos", "neg", "neu", "none"])
        print(f"{a:10s}: đã gán {asp_rows[a].notna().sum()}/200, giá trị sai {wrong.sum()}")
    print(gold[done].sentiment.value_counts())

Chưa có ../labeling/gold_500.csv — gán nhãn xong rồi chạy lại từ đây.


In [5]:
# Bảng chéo nhãn tay × số sao — chứng minh vì sao cần tập vàng (Chương 3)
if os.path.exists(GOLD):
    g = gold[gold.sentiment.isin(["pos", "neg", "neu"])]
    display(pd.crosstab(g.rating, g.sentiment, margins=True))
    display((pd.crosstab(g.rating, g.sentiment, normalize="index") * 100).round(1))
    agree = (g.sentiment == g.weak_label).mean()
    print(f"Nhãn sao trùng với nhãn tay: {agree:.1%}")

In [6]:
# Cohen's kappa giữa 2 lần gán (60 câu)
if os.path.exists(GOLD) and pd.read_csv(RELABEL).sentiment.notna().any():
    r2 = pd.read_csv(RELABEL)[["id", "sentiment"]].merge(gold[["id", "sentiment"]], on="id", suffixes=("_2", "_1"))
    r2 = r2.dropna()
    print(f"{len(r2)} câu, kappa = {cohen_kappa_score(r2.sentiment_1, r2.sentiment_2.str.strip().str.lower()):.3f}")
    display(r2[r2.sentiment_1 != r2.sentiment_2.str.strip().str.lower()])